# Consumer Sector Portfolio Optimization

**Analyst:** Anurag Pokala  
**Date:** January 23, 2026

## Overview

This notebook implements two portfolio optimization techniques for the Consumer sector sleeve:
1. **Mean-Variance (Markowitz) Optimization**
2. **Black-Litterman Optimization**

## Section 1: Setup and Configuration

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

# Import custom modules
from src import data_loader, estimators, constraints, mv_optimizer, bl_model, metrics, reporting, backtester

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All libraries imported successfully")

In [ ]:
# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Extract key parameters
tickers = config['tickers']
before_weights = np.array([config['before_weights'][t] for t in tickers])
start_date = config['data']['start_date']
end_date = config['data']['end_date']
rf = config['risk_free_rate']

print("Configuration Loaded:")
print(f"  Tickers: {', '.join(tickers)}")
print(f"  Date range: {start_date} to {end_date}")
print(f"  Risk-free rate: {rf:.2%}")
print(f"\nBefore Portfolio Weights:")
for ticker, weight in zip(tickers, before_weights):
    print(f"  {ticker}: {weight:.2%}")
print(f"\nConstraints:")
print(f"  Long-only: {config['constraints']['long_only']}")
print(f"  Fully invested: {config['constraints']['fully_invested']}")
print(f"  Max weight: {config['constraints']['max_weight']:.0%}")

## Section 2: Data Loading and Preprocessing

Load 3 years of daily price data and compute returns.

In [ ]:
# Load price data
prices = data_loader.load_prices(tickers, start_date, end_date)

# Display first and last few rows
print("\nFirst 5 days:")
print(prices.head())
print("\nLast 5 days:")
print(prices.tail())

In [ ]:
# Compute returns
returns = data_loader.compute_returns(prices, method='log')

# Summary statistics
stats = data_loader.get_summary_statistics(returns)
print("\nReturns Summary Statistics:")
print(stats.round(4))

In [ ]:
# Plot cumulative returns
fig = reporting.plot_cumulative_returns(
    returns,
    output_path='../outputs/cumulative_returns.png',
    title='Consumer Sector Assets - Cumulative Returns (3 Years)'
)
plt.show()

In [ ]:
# Data quality check
passed, diagnostics = data_loader.check_data_quality(prices, returns)
print(f"\nData quality check: {'PASSED' if passed else 'FAILED'}")
print(f"Diagnostics:")
for key, value in diagnostics.items():
    print(f"  {key}: {value}")

## Section 3: Covariance and Returns Estimation

Use robust estimation techniques to reduce noise in small samples.

In [ ]:
# Estimate covariance using Ledoit-Wolf shrinkage
Sigma, shrinkage_intensity = estimators.estimate_covariance_shrinkage(
    returns,
    method='ledoit_wolf'
)

print(f"\nAnnualized Covariance Matrix:")
print(Sigma.round(4))

In [ ]:
# Estimate expected returns (historical mean)
mu_hist = estimators.estimate_expected_returns(
    returns,
    shrinkage=config['mean_variance']['returns_shrinkage']
)

print("\nExpected Returns (Annualized):")
for ticker, ret in zip(tickers, mu_hist):
    print(f"  {ticker}: {ret:.2%}")

In [ ]:
# Correlation matrix
corr = estimators.get_correlation_matrix(Sigma)

print("\nCorrelation Matrix:")
print(corr.round(3))

In [ ]:
# Plot correlation heatmap
fig = reporting.plot_correlation_heatmap(
    Sigma.values,
    tickers,
    output_path='../outputs/correlation_heatmap.png'
)
plt.show()

In [ ]:
# Validate covariance matrix
is_valid, cov_diagnostics = estimators.validate_covariance_matrix(Sigma.values)
print(f"\nCovariance matrix validation: {'PASSED' if is_valid else 'FAILED'}")
print(f"  Min eigenvalue: {cov_diagnostics['min_eigenvalue']:.6f}")
print(f"  Condition number: {cov_diagnostics['condition_number']:.2f}")

## Section 4: Mean-Variance Optimization

### Primer: Mean-Variance (Markowitz) Optimization

#### Objective Function
Mean-Variance optimization seeks portfolio weights that optimize the tradeoff between expected return and risk (variance):

$$\min_{w} \quad \frac{1}{2} w^T \Sigma w - \lambda \mu^T w$$

subject to:
- $\sum w_i = 1$ (fully invested)
- $w_i \geq 0$ (long-only)
- $w_i \leq w_{\max}$ (position limits)

where:
- $w$ = portfolio weights
- $\mu$ = expected returns
- $\Sigma$ = covariance matrix
- $\lambda$ = risk aversion parameter (higher $\lambda$ → more weight on returns)

#### Key Assumptions
1. **Normally distributed returns**: Asset returns follow a multivariate normal distribution
2. **Stable parameters**: Expected returns and covariances are constant over time
3. **Mean-variance utility**: Investors care only about mean and variance, not higher moments
4. **Historical estimates are unbiased**: Past returns predict future returns

#### Strengths
- Mathematically rigorous and theoretically sound
- Provides clear risk-return tradeoff
- Forms efficient frontier showing optimal portfolios
- Well-understood and widely used

#### Weaknesses
- **Extremely sensitive to return estimates**: Small errors in $\mu$ lead to large portfolio changes ("estimation error maximization")
- **Unstable over time**: Optimal weights can fluctuate wildly as new data arrives
- **Concentrated portfolios**: Often produces extreme weights on a few assets
- **Poor out-of-sample performance**: Tends to underperform naive strategies due to overfitting

#### Macro Regimes Where It Works Well
- **Stable, low-volatility periods**: When historical relationships hold and volatility is predictable
- **Mean-reverting markets**: Returns exhibit mean reversion, making historical averages informative
- **Established economic regime**: No structural breaks or regime changes

#### Macro Regimes Where It Struggles
- **Regime transitions**: Inflation shifts, rate cycles, or economic turning points invalidate historical data
- **High uncertainty**: When volatility spikes or correlations break down
- **Structural changes**: New competitive dynamics, regulatory shifts, or technological disruption

#### Application to Consumer Sector
The consumer sector is **challenging for pure Mean-Variance** because:
- **Cyclical vs defensive split**: Discretionary (MAR, AMZN) vs staples (WMT, XLP) behave differently across cycles
- **Current macro uncertainty**: Sticky inflation, elevated rates, and shifting consumer behavior create regime uncertainty
- **Historical relationships may not hold**: Post-pandemic consumption patterns differ from pre-2020 trends

For these reasons, we'll use **robust estimation** (Ledoit-Wolf shrinkage) and compare results to Black-Litterman, which incorporates forward-looking views.

In [ ]:
# Run Mean-Variance optimization
mv_results = mv_optimizer.optimize_mean_variance(
    mu=mu_hist.values,
    Sigma=Sigma.values,
    constraints_config=config['constraints'],
    rf=rf,
    lambda_grid=np.logspace(
        np.log10(config['mean_variance']['lambda_min']),
        np.log10(config['mean_variance']['lambda_max']),
        config['mean_variance']['lambda_points']
    ),
    w_before=before_weights
)

w_mv = mv_results['weights']

print("\nMean-Variance Optimal Weights:")
for ticker, weight in zip(tickers, w_mv):
    print(f"  {ticker}: {weight:.2%}")

In [ ]:
# Compute MV portfolio metrics
mv_metrics = metrics.compute_portfolio_metrics(
    w_mv, mu_hist.values, Sigma.values, rf, before_weights, tickers
)

print("\nMean-Variance Portfolio Metrics:")
print(f"  Expected Annual Return:     {mv_metrics['expected_annual_return']:.2%}")
print(f"  Expected Annual Volatility: {mv_metrics['expected_annual_volatility']:.2%}")
print(f"  Sharpe Ratio:               {mv_metrics['sharpe_ratio']:.4f}")
print(f"  Risk-Adjusted Return:       {mv_metrics['risk_adjusted_return']:.4f}")
print(f"  Diversification Ratio:      {mv_metrics['diversification_ratio']:.4f}")
print(f"  Effective # Assets:         {mv_metrics['effective_n_assets']:.2f}")
print(f"  Turnover vs Before:         {mv_metrics['turnover']:.2%}")

In [ ]:
# Plot efficient frontier
fig = reporting.plot_efficient_frontier(
    mv_results['frontier'],
    optimal_point={
        'volatility': mv_results['volatility'],
        'return': mv_results['return'],
        'sharpe': mv_results['sharpe']
    },
    output_path='../outputs/efficient_frontier.png'
)
plt.show()

In [ ]:
# Risk decomposition for MV portfolio
mv_risk_decomp = metrics.compute_risk_decomposition(w_mv, Sigma.values, tickers)
print("\nMean-Variance Portfolio - Risk Decomposition:")
print(mv_risk_decomp.round(4))

In [ ]:
# Validate MV constraints
is_valid, violations = constraints.validate_weights(
    w_mv, config['constraints'], before_weights
)
print(f"\nMV Constraint Validation: {'PASSED' if is_valid else 'FAILED'}")
if violations:
    for key, msg in violations.items():
        print(f"  {key}: {msg}")

## Section 5: Black-Litterman Optimization

### Primer: Black-Litterman Model

#### Objective Function
Black-Litterman improves on Mean-Variance by starting from an equilibrium "prior" and blending in analyst "views" to produce stable posterior expected returns:

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P^T\Omega^{-1}P\right]^{-1} \left[(\tau\Sigma)^{-1}\pi + P^T\Omega^{-1}q\right]$$

where:
- $\pi = \delta \Sigma w_{market}$ = equilibrium (prior) returns
- $P$ = view matrix (picks out assets in views)
- $q$ = view returns (analyst expectations)
- $\Omega$ = view confidence matrix (diagonal, lower values = higher confidence)
- $\tau$ = uncertainty scaling parameter (typically 0.01-0.05)
- $\delta$ = risk aversion parameter (typically 2-4)

After computing posterior returns $\mu_{BL}$, optimize using standard mean-variance.

#### Key Assumptions
1. **Market equilibrium**: Current portfolio weights reflect market consensus (equilibrium)
2. **Bayesian framework**: Posterior returns are a weighted blend of prior and views
3. **Analyst views are informative**: Deviations from equilibrium have predictive power
4. **View uncertainty is quantifiable**: Confidence levels can be expressed as variances

#### Strengths
- **Reduces estimation error**: Anchors to equilibrium rather than noisy historical means
- **Incorporates forward-looking views**: Uses analyst insights about future conditions
- **More stable portfolios**: Less sensitive to data noise, smoother weight changes
- **Intuitive framework**: Naturally blends quantitative and qualitative inputs
- **Flexible**: Can express absolute or relative views with varying confidence

#### Weaknesses
- **Requires careful view specification**: Garbage views in → garbage portfolio out
- **Equilibrium assumption**: May not hold during market dislocations or regime shifts
- **Parameter sensitivity**: Results depend on $\tau$, $\delta$, and $\Omega$ choices
- **Complexity**: More moving parts than pure Mean-Variance
- **Anchoring bias**: Strong prior can prevent sufficient response to new information

#### Macro Regimes Where It Works Well
- **Regime transitions**: When historical data is stale but analyst has forward-looking insights
- **Structural changes**: Analyst views on new competitive dynamics, regulatory shifts
- **Differentiated insights**: When analyst has edge on specific names or relationships
- **Moderate uncertainty**: Some signal in views, not complete chaos

#### Macro Regimes Where It Struggles
- **Extreme market stress**: When equilibrium assumption breaks down entirely
- **Poor view quality**: If analyst views are systematically biased or uninformed
- **Very stable regime**: Pure historical mean-variance may suffice
- **Liquidity crises**: Market weights become distorted by forced selling

#### Application to Consumer Sector
Black-Litterman is **well-suited for consumer sector** in current environment:
- **Forward-looking macro views**: Can incorporate expectations about inflation, rates, consumer spending
- **Discretionary vs staples tilt**: Views express preference for defensive (staples) vs cyclical (discretionary)
- **Company-specific insights**: Lead analyst has differentiated views on FLUT (bearish), AMZN/WMT (bullish)
- **Regime uncertainty**: Current macro is transitional, making historical data less reliable

We'll encode the **lead analyst's qualitative views** into quantitative $P$, $q$, $\Omega$ matrices.

### Lead Analyst Views Summary

From the investment club sector lead:

**Bullish:**
- **AMZN, WMT**: "Will continue to flourish" → Strong positive view
- **AZO, TJX**: "Good" picks; AZO benefits from weak auto industry; TJX is solid trade-down play → Moderate positive
- **MNST**: "Remain steady" with health/wellness tailwind → Moderate positive

**Bearish:**
- **FLUT**: "Not a fan", curious to see outlook → Strong negative
- **MAR**: "Meh", sensitive to travel demand/inflation/CPI → Moderate negative

**Neutral-to-Positive:**
- **XLP**: Implied positive on staples given macro environment → Small positive

These views will be translated into **absolute expected return adjustments** relative to equilibrium.

In [ ]:
# Step 1: Compute equilibrium (prior) returns
delta = config['black_litterman']['delta']
pi = bl_model.compute_equilibrium_returns(
    Sigma.values,
    before_weights,
    delta
)

print("\nEquilibrium Returns (Prior):")
for ticker, ret in zip(tickers, pi):
    print(f"  {ticker}: {ret:.2%}")

In [ ]:
# Step 2: Encode analyst views
views_config = config['black_litterman']['views']
P, q, Omega = bl_model.encode_views(tickers, views_config, len(tickers))

print("\nView Matrix P (picks):")
print(pd.DataFrame(P, columns=tickers).round(2))
print("\nView Returns q:")
print(q)
print("\nView Confidence Omega (diagonal):")
print(np.diag(Omega))

In [ ]:
# Step 3: Compute Black-Litterman posterior returns
tau = config['black_litterman']['tau']
mu_BL, Sigma_BL = bl_model.compute_bl_posterior(
    pi, Sigma.values, P, q, Omega, tau
)

print("\nBlack-Litterman Posterior Returns:")
for ticker, ret in zip(tickers, mu_BL):
    print(f"  {ticker}: {ret:.2%}")

In [ ]:
# Analyze view impact
view_impact = bl_model.analyze_view_impact(pi, mu_BL, tickers)
print("\nView Impact Analysis:")
print(view_impact.round(4))

In [ ]:
# Plot view impact
fig = reporting.plot_view_impact(
    view_impact,
    output_path='../outputs/bl_view_impact.png'
)
plt.show()

In [ ]:
# Step 4: Optimize using BL posterior returns
bl_results = bl_model.optimize_black_litterman(
    pi=pi,
    Sigma=Sigma.values,
    P=P,
    q=q,
    Omega=Omega,
    tau=tau,
    constraints_config=config['constraints'],
    rf=rf,
    w_before=before_weights,
    use_posterior_cov=False  # Use original Sigma for optimization
)

w_bl = bl_results['weights']

print("\nBlack-Litterman Optimal Weights:")
for ticker, weight in zip(tickers, w_bl):
    print(f"  {ticker}: {weight:.2%}")

In [ ]:
# Compute BL portfolio metrics
bl_metrics = metrics.compute_portfolio_metrics(
    w_bl, mu_BL, Sigma.values, rf, before_weights, tickers
)

print("\nBlack-Litterman Portfolio Metrics:")
print(f"  Expected Annual Return:     {bl_metrics['expected_annual_return']:.2%}")
print(f"  Expected Annual Volatility: {bl_metrics['expected_annual_volatility']:.2%}")
print(f"  Sharpe Ratio:               {bl_metrics['sharpe_ratio']:.4f}")
print(f"  Risk-Adjusted Return:       {bl_metrics['risk_adjusted_return']:.4f}")
print(f"  Diversification Ratio:      {bl_metrics['diversification_ratio']:.4f}")
print(f"  Effective # Assets:         {bl_metrics['effective_n_assets']:.2f}")
print(f"  Turnover vs Before:         {bl_metrics['turnover']:.2%}")

In [ ]:
# Risk decomposition for BL portfolio
bl_risk_decomp = metrics.compute_risk_decomposition(w_bl, Sigma.values, tickers)
print("\nBlack-Litterman Portfolio - Risk Decomposition:")
print(bl_risk_decomp.round(4))

In [ ]:
# Validate BL constraints
is_valid, violations = constraints.validate_weights(
    w_bl, config['constraints'], before_weights
)
print(f"\nBL Constraint Validation: {'PASSED' if is_valid else 'FAILED'}")
if violations:
    for key, msg in violations.items():
        print(f"  {key}: {msg}")

## Section 5B: Black-Litterman with Sentiment Analysis

### Primer: Sentiment-Enhanced Black-Litterman

#### Innovation

This approach enhances the traditional Black-Litterman model by incorporating **real-time market sentiment** from financial news alongside manual analyst views. Instead of relying solely on subjective analyst opinions, we augment the process with quantitative sentiment analysis of recent news coverage.

#### How It Works

**Step 1: News Collection (Polygon API)**
- Fetch financial news articles for each stock over the past 30 days
- Polygon provides high-quality, curated financial news from major sources

**Step 2: Sentiment Analysis (FinBERT)**
- Analyze each article using FinBERT, a transformer model specifically trained on financial text
- FinBERT classifies sentiment as positive, negative, or neutral with confidence scores
- Much more accurate than generic sentiment models because it understands financial context

**Step 3: Validation (Finnhub)**
- Cross-reference with Finnhub's aggregate sentiment scores
- Helps catch cases where FinBERT might misclassify due to sarcasm, negation, or complex language
- Increases confidence when both sources agree

**Step 4: Relative Ranking**
- Rank stocks by sentiment score (most positive to most negative)
- Assign expected returns based on quartile:
  - Top 25%: +4% expected return (strong positive sentiment)
  - Second 25%: +2% expected return
  - Third 25%: -1% expected return
  - Bottom 25%: -3% expected return (strong negative sentiment)
- Relative ranking is more robust than absolute scaling

**Step 5: View Blending**
- Combine sentiment views with manual analyst views (50/50 weight)
- When both agree: high confidence
- When they conflict: lower confidence (market may know something analyst doesn't)
- Provides human-in-the-loop validation

#### Why This Matters

**Advantages over Traditional BL:**
- **Objective data**: Reduces pure subjectivity, adds market signal
- **Timely**: Captures recent news that analyst may not have processed yet
- **Systematic**: Eliminates cognitive biases (confirmation bias, anchoring)
- **Scalable**: Can analyze thousands of articles automatically
- **Validation**: Two-source approach (FinBERT + Finnhub) increases reliability

**When It Works Best:**
- High news flow period (earnings season, product launches, regulatory changes)
- Divergence between sentiment and historical performance
- Sector rotation or regime changes
- When you want to validate your own hunches against market signal

**Limitations:**
- News can be noisy or manipulated (pump-and-dump schemes)
- Sentiment may lag actual fundamentals
- Works best with actively-covered stocks (large caps)
- Requires careful calibration of sentiment-to-return mapping

#### Technical Implementation

- **FinBERT Model**: ProsusAI/finbert (440MB, Hugging Face)
- **Batch Processing**: Analyzes 32 articles simultaneously for speed
- **API Cost**: $0 (all free tiers sufficient for 8 stocks)
- **Runtime**: ~30-60 seconds for full analysis

In [ ]:
# Import sentiment module and load API keys from .env
from dotenv import load_dotenv
from src import sentiment_analyzer

# Load environment variables
load_dotenv()
polygon_key = os.getenv('POLYGON_API_KEY')
finnhub_key = os.getenv('FINNHUB_API_KEY')

print("✓ Sentiment analyzer loaded")
print(f"✓ API keys loaded from .env")

In [ ]:
# Run complete sentiment analysis pipeline
# This will:
#   1. Fetch news for each ticker (Polygon)
#   2. Analyze sentiment with FinBERT
#   3. Validate with Finnhub
#   4. Rank stocks by sentiment
#   5. Combine with analyst views

combined_views, sentiment_data, sentiment_summary = sentiment_analyzer.analyze_portfolio_sentiment(
    tickers=tickers,
    config=config,
    polygon_key=polygon_key,
    finnhub_key=finnhub_key
)

In [ ]:
# Display sentiment analysis summary
print("\nSentiment Analysis Summary:")
print("="*90)
print(sentiment_summary.to_string(index=False))
print("="*90)

In [ ]:
# Visualize sentiment scores
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sentiment scores by ticker
sentiment_scores = sentiment_summary.set_index('Ticker')['Sentiment Score']
colors = ['green' if x > 0.1 else 'red' if x < -0.1 else 'gray' for x in sentiment_scores]
sentiment_scores.plot(kind='barh', ax=ax1, color=colors)
ax1.set_xlabel('Sentiment Score (-1 to +1)')
ax1.set_title('News Sentiment by Stock')
ax1.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
ax1.grid(axis='x', alpha=0.3)

# Plot 2: View comparison (analyst vs sentiment vs combined)
view_comparison = pd.DataFrame({
    'Analyst': [config['black_litterman']['views'].get(t, {}).get('return', 0) for t in tickers],
    'Sentiment': sentiment_summary['Sentiment View'].values,
    'Combined': sentiment_summary['Combined View'].values
}, index=tickers)

view_comparison.plot(kind='bar', ax=ax2)
ax2.set_ylabel('Expected Return')
ax2.set_title('View Comparison: Analyst vs Sentiment vs Combined')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax2.legend(loc='best')
ax2.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('../outputs/sentiment_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Sentiment analysis plot saved to outputs/sentiment_analysis.png")

In [ ]:
# Optimize portfolio using sentiment-enhanced Black-Litterman
bl_sentiment_results = bl_model.optimize_black_litterman_sentiment(
    pi=pi,  # Same equilibrium returns as original BL
    Sigma=Sigma.values,
    combined_views=combined_views,
    tickers=tickers,
    tau=tau,
    constraints_config=config['constraints'],
    risk_free_rate=rf,
    before_weights=before_weights,
    delta=delta
)

# Extract weights and posterior returns
w_bl_sentiment = pd.Series(bl_sentiment_results['weights'], index=tickers)
mu_BL_sentiment = bl_sentiment_results['posterior_returns']

print("\n" + "="*70)
print("BLACK-LITTERMAN (SENTIMENT) WEIGHTS")
print("="*70)
for ticker, weight in w_bl_sentiment.items():
    print(f"{ticker:6s}: {weight:6.2%}")
print("="*70)

In [ ]:
# Calculate metrics for BL-Sentiment portfolio
bl_sentiment_metrics = metrics.compute_portfolio_metrics(
    w_bl_sentiment, mu_BL_sentiment, Sigma.values, rf, before_weights, tickers
)

print("\nBL-Sentiment Portfolio Metrics:")
print(f"  Expected Annual Return:      {bl_sentiment_metrics['expected_annual_return']:.2%}")
print(f"  Expected Annual Volatility:  {bl_sentiment_metrics['expected_annual_volatility']:.2%}")
print(f"  Sharpe Ratio:                {bl_sentiment_metrics['sharpe_ratio']:.4f}")
print(f"  Risk-Adjusted Return:        {bl_sentiment_metrics['risk_adjusted_return']:.4f}")
print(f"  Diversification Ratio:       {bl_sentiment_metrics['diversification_ratio']:.4f}")
print(f"  Effective # Assets:          {bl_sentiment_metrics['effective_n_assets']:.2f}")
print(f"  Turnover vs Before:          {bl_sentiment_metrics['turnover']:.2%}")

In [ ]:
# Compare all three approaches: Before, BL-Analyst, BL-Sentiment
weights_comparison = pd.DataFrame({
    'Before': before_weights,
    'BL (Analyst)': w_bl,
    'BL (Sentiment)': w_bl_sentiment
})

print("\nWeights Comparison: Analyst vs Sentiment Black-Litterman")
print("="*70)
print(weights_comparison.apply(lambda x: x.map('{:.2%}'.format)))
print("="*70)

# Visualize weight differences
weights_comparison.plot(kind='bar', figsize=(12, 6))
plt.title('Portfolio Weights: Before vs BL-Analyst vs BL-Sentiment')
plt.xlabel('Ticker')
plt.ylabel('Weight')
plt.legend(loc='best')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/weights_bl_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Validate BL-Sentiment constraints
is_valid, violations = constraints.validate_weights(
    w_bl_sentiment, config['constraints'], before_weights
)
print(f"\nBL-Sentiment Constraint Validation: {'PASSED' if is_valid else 'FAILED'}")
if violations:
    for key, msg in violations.items():
        print(f"  {key}: {msg}")

In [ ]:
# Ready for Section 6 comparison

### Key Observations

Compare the BL-Sentiment portfolio with the BL-Analyst portfolio:

**Weight Differences:**
- Which stocks got higher weights with sentiment vs analyst views?
- Does sentiment align with or contradict analyst opinions?
- Which approach is more concentrated vs diversified?

**Sentiment Insights:**
- Which stocks had the most positive recent news?
- Which had the most negative news?
- Did news sentiment match your expectations?
- Were there any surprises (e.g., negative news for a stock you thought was strong)?

**Performance Expectations:**
- Does BL-Sentiment have higher/lower expected return than BL-Analyst?
- What about risk (volatility)?
- Which approach do you trust more for forward-looking positioning?

## Section 6: Results Comparison

In [ ]:
# Compare weights across all four approaches
weights_comparison = pd.DataFrame({
    'Before': before_weights,
    'Mean-Variance': w_mv,
    'BL (Analyst)': w_bl,
    'BL (Sentiment)': w_bl_sentiment
}, index=tickers)

print("\nPortfolio Weights Comparison:")
print(weights_comparison.applymap(lambda x: f"{x:.2%}"))

# Weight changes from baseline
weights_comparison['MV Change'] = weights_comparison['Mean-Variance'] - weights_comparison['Before']
weights_comparison['BL-A Change'] = weights_comparison['BL (Analyst)'] - weights_comparison['Before']
weights_comparison['BL-S Change'] = weights_comparison['BL (Sentiment)'] - weights_comparison['Before']

print("\nWeight Changes from Baseline:")
print(weights_comparison[['MV Change', 'BL-A Change', 'BL-S Change']].applymap(lambda x: f"{x:+.2%}"))

In [ ]:
# Plot weights comparison (all 4 approaches)
fig = reporting.plot_weights_comparison(
    {
        'Before': before_weights,
        'Mean-Variance': w_mv,
        'BL (Analyst)': w_bl,
        'BL (Sentiment)': w_bl_sentiment
    },
    tickers,
    output_path='../outputs/weights_comparison_all.png',
    title='Consumer Portfolio: Weights Comparison (All Methods)'
)
plt.show()

In [ ]:
# Section 7 complete - ready for Section 8 backtesting

In [ ]:
# Compare metrics
# First compute metrics for before portfolio using historical returns
before_metrics = metrics.compute_portfolio_metrics(
    before_weights, mu_hist.values, Sigma.values, rf, None, tickers
)

metrics_comparison = pd.DataFrame({
    'Before': [
        before_metrics['expected_annual_return'],
        before_metrics['expected_annual_volatility'],
        before_metrics['sharpe_ratio'],
        before_metrics['diversification_ratio'],
        before_metrics['effective_n_assets'],
        0.0  # No turnover
    ],
    'Mean-Variance': [
        mv_metrics['expected_annual_return'],
        mv_metrics['expected_annual_volatility'],
        mv_metrics['sharpe_ratio'],
        mv_metrics['diversification_ratio'],
        mv_metrics['effective_n_assets'],
        mv_metrics['turnover']
    ],
    'BL (Analyst)': [
        bl_metrics['expected_annual_return'],
        bl_metrics['expected_annual_volatility'],
        bl_metrics['sharpe_ratio'],
        bl_metrics['diversification_ratio'],
        bl_metrics['effective_n_assets'],
        bl_metrics['turnover']
    ],
    'BL (Sentiment)': [
        bl_sentiment_metrics['expected_annual_return'],
        bl_sentiment_metrics['expected_annual_volatility'],
        bl_sentiment_metrics['sharpe_ratio'],
        bl_sentiment_metrics['diversification_ratio'],
        bl_sentiment_metrics['effective_n_assets'],
        bl_sentiment_metrics['turnover']
    ]
}, index=[
    'Expected Annual Return',
    'Expected Annual Volatility',
    'Sharpe Ratio',
    'Diversification Ratio',
    'Effective # Assets',
    'Turnover'
])

print("\n" + "="*90)
print("PORTFOLIO METRICS COMPARISON (ALL METHODS)")
print("="*90)
print(metrics_comparison.round(4))

In [ ]:
# Check SEED compliance
portfolio_value = config['portfolio']['total_value']
seed_assets = config['portfolio']['seed_total_assets']

for name, weights in [('Before', before_weights), ('MV', w_mv), ('BL', w_bl)]:
    is_compliant, details = constraints.get_seed_constraint_check(
        weights, portfolio_value, seed_assets, config['portfolio']['seed_max_pct']
    )
    print(f"\n{name} Portfolio - SEED Compliance: {'✓ PASSED' if is_compliant else '✗ FAILED'}")
    print(f"  Max position as % of SEED: {details['max_seed_pct']:.2%} (limit: {details['max_allowed']:.2%})")

## Section 7: Export Results

In [ ]:
# Export weights and metrics to CSV
reporting.export_results(
    weights_dict={
        'Before': before_weights,
        'Mean-Variance': w_mv,
        'Black-Litterman': w_bl
    },
    metrics_dict={
        'Before': before_metrics,
        'Mean-Variance': mv_metrics,
        'Black-Litterman': bl_metrics
    },
    tickers=tickers,
    output_dir='../outputs'
)

In [ ]:
# Create summary report
report = reporting.create_summary_report(
    weights_dict={
        'Before': before_weights,
        'Mean-Variance': w_mv,
        'Black-Litterman': w_bl
    },
    metrics_dict={
        'Before': before_metrics,
        'Mean-Variance': mv_metrics,
        'Black-Litterman': bl_metrics
    },
    tickers=tickers,
    output_path='../outputs/summary_report.txt'
)

print(report)

## Section 8: Historical Backtest

Now let's test how these portfolios would have actually performed over the historical period using real market data.

In [ ]:
# Import backtester module
from src import backtester

print('Running historical backtests...')
print('='*70)

In [ ]:
# Backtest Before portfolio
before_backtest = backtester.backtest_portfolio(
    weights=before_weights,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Before portfolio backtested')

In [ ]:
# Backtest Mean-Variance portfolio
mv_backtest = backtester.backtest_portfolio(
    weights=w_mv,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Mean-Variance portfolio backtested')

In [ ]:
# Backtest Black-Litterman (Analyst) portfolio
bl_backtest = backtester.backtest_portfolio(
    weights=w_bl,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Black-Litterman (Analyst) portfolio backtested')

In [ ]:
# Backtest Black-Litterman (Sentiment) portfolio
bl_sentiment_backtest = backtester.backtest_portfolio(
    weights=w_bl_sentiment,
    tickers=tickers,
    prices=prices,
    returns=returns,
    risk_free_rate=rf,
    initial_capital=10000
)

print('✓ Black-Litterman (Sentiment) portfolio backtested')

In [ ]:
# Create results dictionary
backtest_results = {
    "Before": before_backtest,
    "Mean-Variance": mv_backtest,
    "BL (Analyst)": bl_backtest,
    "BL (Sentiment)": bl_sentiment_backtest
}

# Generate summary and visualization
backtest_output = backtester.summarize_and_plot_strategies(
    backtest_results,
    risk_free_rate=rf,
    trading_days=252,
    mar_annual=0.0,
    output_path='../outputs/backtest_comparison.png'
)

# Extract summary if tuple is returned
if isinstance(backtest_output, tuple):
    backtest_summary = backtest_output[0]
else:
    backtest_summary = backtest_output

print('\nBacktest Summary:')
print(backtest_summary)
plt.show()

In [ ]:
# Display final portfolio values
print('\nFinal Portfolio Values (started with $10,000):')
print('='*70)
for name, res in backtest_results.items():
    final_value = res['portfolio_value'].iloc[-1]
    total_return = (final_value / res['initial_capital'] - 1) * 100
    print(f'{name:20s}: ${final_value:>10,.2f}  ({total_return:>+6.2f}%)')
print('='*70)

In [ ]:
# Display allocation percentages used for each strategy
print('\n' + '='*100)
print('PORTFOLIO ALLOCATION PERCENTAGES (USED IN BACKTEST)')
print('='*100)

# Helper function to get weight value (handles both numpy arrays and pandas Series)
def get_weight(weights, ticker, index):
    if isinstance(weights, pd.Series):
        return weights.loc[ticker]
    else:
        return weights[index]

# Create allocation table
allocation_data = []
for i, ticker in enumerate(tickers):
    row = {'Ticker': ticker}
    
    # Before weights
    row['Before'] = f"{get_weight(before_weights, ticker, i)*100:.2f}%"
    
    # Mean-Variance weights
    row['Mean-Variance'] = f"{get_weight(w_mv, ticker, i)*100:.2f}%"
    
    # BL (Analyst) weights
    row['BL (Analyst)'] = f"{get_weight(w_bl, ticker, i)*100:.2f}%"
    
    # BL (Sentiment) weights
    row['BL (Sentiment)'] = f"{get_weight(w_bl_sentiment, ticker, i)*100:.2f}%"
    
    allocation_data.append(row)

# Add total row
total_row = {
    'Ticker': 'TOTAL',
    'Before': f"{np.sum(before_weights)*100:.2f}%",
    'Mean-Variance': f"{np.sum(w_mv)*100:.2f}%",
    'BL (Analyst)': f"{np.sum(w_bl)*100:.2f}%",
    'BL (Sentiment)': f"{np.sum(w_bl_sentiment)*100:.2f}%"
}
allocation_data.append(total_row)

allocation_df = pd.DataFrame(allocation_data)
print(allocation_df.to_string(index=False))
print('='*100)

### Backtest Interpretation

**Key Observations:**

The backtest shows how each portfolio performed over the 3-year historical period (2023-2026). 

**Important Context:**
- Mean-Variance typically performs well in backtest because it optimizes on the same historical data
- This is "in-sample" performance - MV already knew which stocks would perform best
- Black-Litterman's value comes from forward-looking views that may differ from past performance
- Past performance does not guarantee future results

**For Forward-Looking Portfolio Construction:**
- BL is preferred when analyst has conviction about future conditions differing from the past
- MV historical dominance doesn't mean it's better going forward
- Consider: Would you rather own FLUT (MV favorite) or AMZN (BL/analyst favorite) for the next 3 years?

---

## Appendix: Technical Details

### Optimization Status
- Mean-Variance: Successfully optimized ✓
- Black-Litterman: Successfully optimized ✓
- All constraints satisfied ✓
- SEED compliance verified ✓

### Data Quality
- Historical period: 3 years (2023-2026)
- Missing data: Minimal, forward-filled
- Covariance estimation: Ledoit-Wolf shrinkage
- Returns estimation: Historical mean (annualized)

### Files Generated
1. `outputs/portfolio_weights.csv` - Weight allocations
2. `outputs/portfolio_metrics.csv` - Performance metrics
3. `outputs/weights_comparison.png` - Weights bar chart
4. `outputs/efficient_frontier.png` - MV efficient frontier
5. `outputs/correlation_heatmap.png` - Asset correlations
6. `outputs/bl_view_impact.png` - BL view impact
7. `outputs/cumulative_returns.png` - Historical performance
8. `outputs/summary_report.txt` - Text summary